In [ ]:
!pip install --upgrade torch torchvision torchaudio transformers accelerate --extra-index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu130


# Login to HF

In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged into Hugging Face!")
except userdata.SecretNotFoundError:
    print("HF_TOKEN not found in Colab secrets. Please add it via the 🔑 tab.")

Successfully logged into Hugging Face!


# Load model

In [ ]:
import os
from transformers import AutoProcessor, AutoModelForCausalLM
import torch

# Download model
MODEL_ID = "google/gemma-4-E2B-it"

# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="cuda:0"
)

print(next(model.parameters()).device)

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

cuda:0


In [ ]:
# Ensure special tokens are properly set
if hasattr(processor, 'tokenizer'):
    tokenizer = processor.tokenizer
    if tokenizer.pad_token_id is not None:
        model.generation_config.pad_token_id = tokenizer.pad_token_id
    if tokenizer.eos_token_id is not None:
        model.generation_config.eos_token_id = tokenizer.eos_token_id

In [ ]:
import sys

def chat(user_message, history=None):
    """Send a message and get a response."""
    if history is None:
        history = [{"role": "system", "content": "You are a helpful assistant."}]

    # Append the new user message
    history.append({"role": "user", "content": user_message})

    # Process input
    text = processor.apply_chat_template(
        history,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    # Generate output
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
    response_raw = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

    # Parse output
    result = processor.parse_response(response_raw)

    # Append assistant reply to history for multi-turn context
    history.append({"role": "assistant", "content": result})

    return result, history


# --- Interactive loop ---
print("Chat started. Type 'quit' or 'exit' to stop.\n")

conversation_history = None  # Reuse across turns for multi-turn memory

while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nExiting.")
        sys.exit(0)

    if not user_input:
        continue
    if user_input.lower() in {"quit", "exit"}:
        print("Goodbye!")
        break

    response, conversation_history = chat(user_input, conversation_history)
    print(f"Assistant: {response.get("content")}\n")

Chat started. Type 'quit' or 'exit' to stop.

You: Tell me about water?
Assistant: That is a wonderful and incredibly broad topic! Water ($\text{H}_2\text{O}$) is arguably the most important substance on Earth, playing a fundamental role in all known life and shaping our planet.

To give you a good overview, I can cover several different aspects of water. Which area interests you most, or would you like a general overview?

Here is a comprehensive summary covering the **basics, properties, importance, and cycles of water**:

---

## 1. The Basics: What is Water?

* **Chemical Formula:** $\text{H}_2\text{O}$ (Two hydrogen atoms bonded to one oxygen atom).
* **Molecular Structure:** Water is a **polar molecule**. This means the oxygen atom pulls the electrons more strongly than the hydrogen atoms, giving the oxygen end a slight negative charge ($\delta-$) and the hydrogen ends slight positive charges ($\delta+$). This polarity allows water molecules to form **hydrogen bonds** with each o

# Saving notebook to github repo

In [ ]:
import os

# Mount drive first if not already
from google.colab import drive
drive.mount('/content/drive')

# Find your notebook
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.ipynb'):
            print(os.path.join(root, file))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/Gemma4 App POC.ipynb


In [ ]:
# Run this cell, then File > Save > push to GitHub
import json, pathlib

nb_path = "/content/drive/MyDrive/Colab Notebooks/Gemma4 App POC.ipynb"  # adjust path

with open(nb_path) as f:
    nb = json.load(f)

# Remove the broken widgets metadata
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open(nb_path, "w") as f:
    json.dump(nb, f, indent=1)

print("Done! Notebook cleaned.")

Done! Notebook cleaned.


In [17]:
from google.colab import files
files.download(nb_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>